# LipSyncNet — Model Architecture Notes

This notebook documents the architecture design and construction plan for **LipSyncNet**, a lip-reading model combining a 3D-CNN frontend, EfficientNetB0 feature extraction, and a Bi-LSTM backend with CTC decoding.

---

## Overview

The model is composed of three major components:

1. **3D-CNN Frontend** — Extracts spatiotemporal features from raw video frames
2. **EfficientNetB0 Branch** — Extracts high-level semantic features per frame (frozen, ImageNet weights)
3. **Bi-LSTM Backend** — Models temporal sequence dependencies for character prediction

These components are combined and trained end-to-end using **CTC (Connectionist Temporal Classification) loss**.

---

> **Design idea:** Set up each component as a separate function and combine them in a single model assembly script.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torch.optim import Adam

In [2]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

In [3]:
NUM_FRAMES      = 75                # temporal depth of each video clip
FRAME_H         = 46                # frame height (pixels)
FRAME_W         = 140               # frame width  (pixels)
IN_CHANNELS     = 1                 # grayscale input

VOCAB_SIZE      = 40                # distinct characters
BLANK_INDEX     = 0                 # CTC blank token index (PyTorch CTCLoss convention: blank=0)
NUM_CLASSES     = VOCAB_SIZE + 1    # 41 output units (incl. blank)

EFFICIENTNET_H  = 224
EFFICIENTNET_W  = 224

LSTM_UNITS      = 512               # units per direction; 512×2 = 1024 output
DROPOUT_RATE    = 0.5
LEARNING_RATE   = 1e-4
BEAM_WIDTH      = 100

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
print(f'PyTorch version: {torch.__version__}')
print(f'Device: {DEVICE}')

PyTorch version: 2.10.0+cpu
Device: cpu


## 1. 3D-CNN Frontend

The 3D-CNN frontend processes raw video clips and extracts spatiotemporal features across frames.

**Input shape:** `(batch, 75, 46, 140, 1)` — 75 frames, 46×140 pixels, 1 grayscale channel

---

### Layer 1 — First Convolutional Block

| Component | Config |
|---|---|
| `Conv3D` | 128 filters, kernel `(3, 5, 5)`, stride `(1,1,1)`, padding `same`, ReLU |
| `BatchNormalization` | Normalize activations |
| `MaxPooling3D` | pool_size `(1, 2, 2)` — spatial halving only |

- Kernel `(3, 5, 5)`: captures motion across 3 frames and facial features in 5×5 spatial patches
- No temporal pooling — preserves all 75 time steps throughout the CNN

**Shape after Layer 1:** `(batch, 75, 23, 70, 128)`

---

### Layer 2 — Second Convolutional Block

| Component | Config |
|---|---|
| `Conv3D` | 256 filters, kernel `(3, 5, 5)`, padding `same`, ReLU |
| `BatchNormalization` | — |
| `MaxPooling3D` | pool_size `(1, 2, 2)` |

**Shape after Layer 2:** `(batch, 75, 11, 35, 256)`

---

### Layer 3 — Third Convolutional Block

| Component | Config |
|---|---|
| `Conv3D` | **75 filters**, kernel `(3, 3, 3)`, padding `same`, ReLU |
| `BatchNormalization` | — |
| `MaxPooling3D` | pool_size `(1, 2, 2)` |

- Filter count reduced to 75 to match the temporal dimension

**Shape after Layer 3:** `(batch, 75, 5, 17, 75)`

---

### Layer 4 — Fourth Convolutional Block

| Component | Config |
|---|---|
| `Conv3D` | 75 filters, kernel `(1, 3, 3)`, padding `same`, ReLU |
| `BatchNormalization` | — |
| `MaxPooling3D` | pool_size `(1, 2, 2)` |

- Temporal kernel size = 1: focuses purely on spatial features at this stage

**Final 3D-CNN output shape:** `(batch, 75, 2, 8, 75)`

---

### Verification Checklist

- [x] Test with dummy input `(1, 75, 46, 140, 1)`
- [x] Print intermediate shapes after each block
- [x] Count parameters in 3D-CNN portion
- [ ] Visualize architecture diagram

In [5]:
class conv3d_block(nn.Module):
    """
    A single 3D convolutional block: Conv3D --> BatchNorm --> MaxPool3D.
    """
    def __init__(self, in_channels: int, out_channels: int,
                 kernel_size: tuple, pool_size: tuple = (1, 2, 2)):
        super().__init__()

        # padding='same' equivalent: pad = kernel // 2 for each dim
        pad = tuple(k // 2 for k in kernel_size)

        self.conv  = nn.Conv3d(in_channels, out_channels,
                               kernel_size=kernel_size,
                               stride=1, padding=pad, bias=False)

        self.bn    = nn.BatchNorm3d(out_channels)

        self.pool  = nn.MaxPool3d(kernel_size=pool_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.pool(F.relu(self.bn(self.conv(x))))

In [6]:
class build_3dcnn_frontend(nn.Module):
    """
    Builds the full 4-block 3D-CNN frontend.

    Shape progression:
        Input  -> (batch, 75, 46, 140,  1)
        Block1 -> (batch, 75, 23,  70, 128), kernel (3,5,5), 128 filters
        Block2 -> (batch, 75, 11,  35, 256), kernel (3,5,5), 256 filters
        Block3 -> (batch, 75,  5,  17,  75), kernel (3,3,3),  75 filters
        Block4 -> (batch, 75,  2,   8,  75), kernel (1,3,3),  75 filters

    Args:
        input_tensor: Keras tensor of shape (batch, T, H, W, 1).

    Returns:
        Output tensor of shape (batch, 75, 2, 8, 75).
    """
    def __init__(self):
        super().__init__()
        self.block1 = conv3d_block(1,   128, (3, 5, 5))
        self.block2 = conv3d_block(128, 256, (3, 5, 5))
        self.block3 = conv3d_block(256,  75, (3, 3, 3))
        self.block4 = conv3d_block(75,   75, (1, 3, 3))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)

        return x  # (B, 75, 75, 2, 8)

In [7]:
cnn = build_3dcnn_frontend().to(DEVICE)

dummy = torch.randn(1, IN_CHANNELS, NUM_FRAMES, FRAME_H, FRAME_W).to(DEVICE)

out = cnn(dummy)

print(f'3D-CNN output shape: {tuple(out.shape)}')

total_cnn = sum(p.numel() for p in cnn.parameters())
print(f'3D-CNN parameters: {total_cnn:,}')

3D-CNN output shape: (1, 75, 75, 2, 8)
3D-CNN parameters: 3,037,293


## 2. EfficientNetB0 Integration

EfficientNetB0 is used as a **fixed feature extractor** per frame, running in parallel with the 3D-CNN features. Its weights are frozen (not fine-tuned), following the paper's approach.

---

### Preparing 3D-CNN Output for EfficientNetB0

EfficientNetB0 expects RGB images of size `224×224`. The 3D-CNN output needs reshaping:

| Step | Shape |
|---|---|
| 3D-CNN output | `(batch, 75, 2, 8, 75)` |
| Target (EfficientNet input) | `(batch, 75, 224, 224, 3)` |

**Conversion steps (applied per frame via `TimeDistributed`):**

1. **Channel expansion:** Repeat grayscale channel 3× → `(2, 8, 75, 3)` per frame  
2. **Resize:** Bilinear interpolation to `224×224` → `(224, 224, 3)` per frame

---

### EfficientNetB0 Configuration

| Parameter | Value |
|---|---|
| `include_top` | `False` (remove classification head) |
| `weights` | `'imagenet'` (pre-trained) |
| `pooling` | `'avg'` (global average pooling) |
| `trainable` | `False` (frozen) |

- Applied per frame using `TimeDistributed`
- Output per frame: `(1280,)` feature vector
- **EfficientNet output shape:** `(batch, 75, 1280)`

---

### Feature Concatenation

The 3D-CNN spatial output is flattened per frame and concatenated with EfficientNet features:

| Feature Source | Shape |
|---|---|
| 3D-CNN (`TimeDistributed(Flatten)`) | `(batch, 75, 1200)` → `2×8×75 = 1200` |
| EfficientNetB0 | `(batch, 75, 1280)` |
| **Concatenated** | **`(batch, 75, 2480)`** |

This combined representation captures both **local spatiotemporal** motion cues (3D-CNN) and **high-level semantic** visual features (EfficientNet).

---

### Verification Checklist

- [ ] Test with dummy input; check feature dimensions
- [ ] Verify EfficientNet weights are frozen
- [ ] Count parameters added by EfficientNet

In [ ]:
class build_efficientnet_branch(nn.Module):
    """
    Frozen EfficientNetB0 feature extractor applied per frame (TimeDistributed).

    Shape progression:
        Input        : (B, T, C, H, W)
        Expand ch    : (B, T, 3, H, W)       grayscale -> 3-ch
        Merge B & T  : (B*T, 3, H, W)
        Resize       : (B*T, 3, 224, 224)
        EfficientNet : (B*T, 1280)            features + avgpool
        Restore T    : (B, T, 1280)

    Args:
        target_h : Target height for EfficientNet input (224).
        target_w : Target width  for EfficientNet input (224).
    """
    def __init__(self, target_h: int = EFFICIENTNET_H,
                 target_w: int = EFFICIENTNET_W):
        super().__init__()
        self.target_h = target_h
        self.target_w = target_w

        # Pretrained EfficientNetB0 (include_top=False, pooling='avg')
        base = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)

        self.features = base.features         # conv backbone
        self.avgpool  = base.avgpool          # AdaptiveAvgPool2d(1)

        # Freeze all weights (trainable=False)
        for param in self.features.parameters():
            param.requires_grad = False
        for param in self.avgpool.parameters():
            param.requires_grad = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C, H, W = x.shape

        if C == 1:
            x = x.expand(B, T, 3, H, W)                          # grayscale -> 3-ch

        x = x.contiguous().reshape(B * T, 3, H, W)               # (B*T, 3, H, W)
        x = F.interpolate(x, size=(self.target_h, self.target_w),
                          mode='bilinear', align_corners=False)   # (B*T, 3, 224, 224)

        with torch.no_grad():
            x = self.features(x)                                  # (B*T, 1280, 7, 7)
            x = self.avgpool(x)                                   # (B*T, 1280, 1, 1)
            x = torch.flatten(x, 1)                               # (B*T, 1280)

        x = x.reshape(B, T, -1)                                  # (B, T, 1280)

        return x

In [ ]:
eff = build_efficientnet_branch().to(DEVICE)

dummy = torch.randn(1, NUM_FRAMES, IN_CHANNELS, FRAME_H, FRAME_W).to(DEVICE)

out = eff(dummy)

print(f'EfficientNet output shape: {tuple(out.shape)}')

all_frozen = all(not p.requires_grad for p in eff.parameters())
print(f'All parameters frozen: {all_frozen}')

total_eff     = sum(p.numel() for p in eff.parameters())
trainable_eff = sum(p.numel() for p in eff.parameters() if p.requires_grad)
print(f'EfficientNet total parameters:     {total_eff:,}')
print(f'EfficientNet trainable parameters: {trainable_eff:,}')

## 3. Bi-LSTM Backend

The Bi-LSTM backend processes the combined feature sequence `(batch, 75, 2480)` to model temporal dependencies in both directions across the 75 time steps.

---

### Layer 1 — First Bidirectional LSTM

| Component | Config |
|---|---|
| `Bidirectional(LSTM)` | 512 units, `return_sequences=True` |
| Forward LSTM | 512 units |
| Backward LSTM | 512 units |
| Concatenated output | 1024 features per time step |
| `Dropout` | rate = 0.5 |

**Output shape:** `(batch, 75, 1024)`

---

### Layer 2 — Second Bidirectional LSTM

| Component | Config |
|---|---|
| `Bidirectional(LSTM)` | 512 units, `return_sequences=True` |
| `Dropout` | rate = 0.5 |

**Output shape:** `(batch, 75, 1024)`

Dropout (50%) is applied after each Bi-LSTM layer to reduce overfitting. It is only active during training.

In [8]:
class build_bi_lstm_backend(nn.Module):
    """
    Two-layer Bidirectional LSTM backend for temporal sequence modeling.

    Shape progression:
        Input      : (B, 75, 2480)
        Bi-LSTM 1  : (B, 75, 1024)   [512 fwd + 512 bwd]
        Dropout 1  : (B, 75, 1024)
        Bi-LSTM 2  : (B, 75, 1024)
        Dropout 2  : (B, 75, 1024)

    Args:
        input_size  : Feature size of each time step (2480 after fusion).
        lstm_units  : Hidden units per direction (512).
        dropout_rate: Dropout fraction after each LSTM layer (0.5).
    """
    def __init__(self, input_size: int = 2480,
                 lstm_units: int = LSTM_UNITS,
                 dropout_rate: float = DROPOUT_RATE):
        super().__init__()

        # Layer 1: input_size -> 1024 (512 × 2 directions)
        self.bilstm1 = nn.LSTM(input_size=input_size,
                               hidden_size=lstm_units,
                               num_layers=1,
                               bidirectional=True,
                               batch_first=True)
        self.drop1   = nn.Dropout(p=dropout_rate)

        # Layer 2: 1024 -> 1024
        self.bilstm2 = nn.LSTM(input_size=lstm_units * 2,  # 1024 from layer 1
                               hidden_size=lstm_units,
                               num_layers=1,
                               bidirectional=True,
                               batch_first=True)
        self.drop2   = nn.Dropout(p=dropout_rate)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x, _ = self.bilstm1(x)   # (B, 75, 1024)
        x     = self.drop1(x)

        x, _ = self.bilstm2(x)   # (B, 75, 1024)
        x     = self.drop2(x)

        return x                 # (B, 75, 1024)

## 4. Output Layer & CTC Loss

---

### Dense Output Layer

| Parameter | Value |
|---|---|
| Units | `vocab_size + 1 = 41` (40 characters + 1 CTC blank token) |
| Activation | `softmax` |
| Input | `(batch, 75, 1024)` |
| **Output** | **`(batch, 75, 41)`** — probability distribution over characters per time step |

---

### CTC Loss

CTC (Connectionist Temporal Classification) allows training without requiring frame-level label alignment.

**Inputs required:**

| Input | Shape |
|---|---|
| `y_pred` | `(batch, 75, 41)` — model predictions |
| `y_true` | `(batch, max_label_length)` — ground truth labels |
| `input_length` | `(batch,)` — actual prediction length |
| `label_length` | `(batch,)` — actual label length |

**How it works:**
- Computes probabilities over all possible CTC alignments (paths)
- Sums probabilities of all paths that collapse to the correct label
- Minimizes the negative log-likelihood of the correct sequence
- Implemented as a custom Keras layer that automatically adds to the model's total loss

---

### CTC Decoder (Inference)

- **Beam search** decoder with beam width = **100** (per paper specification)
- Considers the top 100 most likely sequences at each decoding step
- More accurate than greedy decoding
- Returns the most likely character sequence

## 5. Complete Model Assembly

---

### Architecture Summary

```
Input: (batch, 75, 46, 140, 1)
        │
        ├──── 3D-CNN Frontend ──────────────► (batch, 75, 2, 8, 75)
        │         │                                    │
        │         │ TimeDistributed(Flatten)           │
        │         ▼                                    │
        │     (batch, 75, 1200)                        │
        │                                             │
        └──── EfficientNetB0 Branch ────────► (batch, 75, 1280)
                  (TimeDistributed, frozen)
                         │
                  Concatenate ──────────────► (batch, 75, 2480)
                         │
                  Bi-LSTM ×2 ───────────────► (batch, 75, 1024)
                         │
                  Dense (softmax) ──────────► (batch, 75, 41)
                         │
                  CTC Loss Layer
```

**Model name:** `"LipSyncNet"`  
**Expected total parameters:** ~307M

---

### Training Wrapper

A separate training model wraps the inference model with additional CTC inputs:

| Additional Input | Shape |
|---|---|
| `labels` | `(None,)` — variable length |
| `input_length` | `(1,)` |
| `label_length` | `(1,)` |

---

### Verification Checklist

- [ ] Print full model summary
- [ ] Confirm total parameters ~307M
- [ ] Verify trainable vs. non-trainable parameter split
- [ ] Check all layer names and connections
- [ ] Visualize with `plot_model`
- [ ] Test with dummy data `(2, 75, 46, 140, 1)` — output should be `(2, 75, 41)`

## 6. Model Compilation

---

### Optimizer

| Parameter | Value |
|---|---|
| Optimizer | `Adam` |
| Learning rate | `0.0001` |
| Beta 1 | `0.9` (default) |
| Beta 2 | `0.999` (default) |
| Epsilon | `1e-7` (default) |

---

### Loss & Metrics

- **Loss:** CTC loss is integrated directly into the model via the custom CTC layer — `loss=None` in `compile()`
- **Training metrics:** Loss tracking only during training
- **Custom metrics to consider:**
  - CTC loss value
  - Learning rate (if using a schedule)
- **Validation metrics:** Computed separately (e.g., WER, CER)

---